# Importación de los datos
Importación del dataset y lectura de sus características

In [ ]:
import pandas as pd

DATOS = pd.read_csv("housing.csv")

In [ ]:
DATOS.head()

In [ ]:
DATOS.describe()

In [ ]:
DATOS.info()

In [ ]:
DATOS.isnull().sum()

In [ ]:
# 2 opciones: Eliminar los valores nulos o replazarlos con la media
data_non_null = DATOS.dropna()
data_non_null.info()

data_clean = DATOS.copy()
data_clean["total_bedrooms"] = data_clean["total_bedrooms"].fillna(value=data_clean["total_bedrooms"].mean())

#  Imprimimos para ver si la media cambia:
# Parece que 'total_bedrooms' no cambia (Si cambias los datos nulos por la media no alteras la media global)
# Pero otros valores si pueden cambiar (Si eliminas datos con 'total_bedrooms' nulos, realmente estás borrando
# esa fila de datos entera, por lo que eliminas ese valor del conteo de la media)
print("\ntotal_bedrooms:")
print(f"Media sin valores nulos:  {data_non_null["total_bedrooms"].mean()}")
print(f"Media imputando la media: {data_clean["total_bedrooms"].mean()}")

print("\nhousing_median_age:")
print(f"Media sin valores nulos:  {data_non_null["housing_median_age"].mean()}")
print(f"Media imputando la media: {data_clean["housing_median_age"].mean()}")

# Visualización de los datos
Mediante matplotlib y pandas

In [ ]:
# Elijo quedarme con el dataframe sin datos nulos (y sin imputación de datos)
data = data_non_null

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Historiograma ---
plt.style.use('_mpl-gallery')
# Creamos los ejes de la figura
fig, ax = plt.subplots(figsize=(8, 8))
"""
nrows, ncols: Número de subplots en cada línea/columna 
"""

# Elejimos que datos mostrar
x = np.array(DATOS["total_rooms"])

# Dibujamos el historiograma
ax.hist(x, linewidth=0.5, edgecolor="white", bins=100)
"""
bins: Número de barras
linewidth, edgecolor: Cambiar gráficos
range [tuple]: Indica el rango de los datos 
"""

plt.show()

In [ ]:
# --- Dibujamos varios gráficos ---

def plotGraphs(data):
    """
    Crea 4 gráficos para el conjunto de datos especificado. No muestra datos muy relevantes, simplemente prueba a mostrar datos en diferentes plots
    """
    
    plt.style.use('_mpl-gallery')
    # Creamos los ejes de la figura
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2, figsize=(10, 7), layout="constrained")
    """
    nrows, ncols: Número de subplots en cada línea/columna 
    """

    # Elejimos que datos mostrar
    y = np.array(data["median_house_value"])
    x = np.linspace(0, len(y), len(y))

    # Dibujamos cada gráfico
    ax1.set_title("median_house_value")
    ax1.hist(y, bins=20, linewidth=0.5, edgecolor="white",color="green")

    ax2.set_title("latitude vs longitude")
    ax2.scatter(np.array(data["latitude"]), np.array(data["longitude"]), color="green")
    # ax2.scatter(
    #     np.array(data["median_house_value"]),
    #     np.array(data["total_rooms"]),
    #     color="green",
    # )

    ax3.set_title("median_house_value")
    population = data["population"]
    boxplot_data = [np.multiply(y, 0.01), population]
    labels = ["Median house value (x0.01)", "Population"]
    ax3.set_xticklabels(labels)
    ax3.boxplot(boxplot_data, showmeans=True, showfliers=True)

    ax4.set_title("ocean_proximity")
    x_categorica = data["ocean_proximity"].unique()
    y_categorica = np.array(data["ocean_proximity"].value_counts())
    ax4.bar(x_categorica, y_categorica, linewidth=0.5, color="blue")

    plt.show()
    print("Categóricas:\n", x_categorica, y_categorica)

    print("\nPopulation:")
    print(f"Mean: {population.mean()}, Median: {population.median()}, Min: {population.min()}, Max: {population.max()}")
    print(f"Std: {population.std()}, Var (std^2): {population.var()}")

In [ ]:
plotGraphs(data)

In [ ]:
def plotBoxplots(data, rows=2, cols=5):
    """
    Muestra un boxplot para cada valor numérico de 'data'
    """
    
    
    # Creamos los ejes de la figura
    plot_data = data.select_dtypes(include="number")

    fig, axs = plt.subplots(nrows=rows, ncols=cols, figsize=(16, 6), layout="constrained")
    # Aplanamos la matriz para recorrer cada uno de sus valores (no nos interesan sus índices x,y)
    axs = axs.flatten()

    for i, col in enumerate(plot_data.columns):
        ax = axs[i]
        ax.set_title(col)
        data = np.array(plot_data[col])
        ax.boxplot(data, showmeans=True, showfliers=True)
        
    # Ocultamos las celdas vacías (si las hay)
    for j in range(i + 1, len(axs)):
        axs[j].set_visible(False)

    plt.show()

In [ ]:
plotBoxplots(data)

In [ ]:
# Algunos tests para ver que valores son atípicos y cuales no

habitaciones = data["total_rooms"] / data["households"]
print("mediana", habitaciones.mean())

Q1, Q3 = habitaciones.quantile([0.25, 0.75])
rango_intercuartil = Q3 - Q1
print(Q1, Q3, rango_intercuartil)
print("outliers", Q3 + rango_intercuartil * 1.5)

fig, ax = plt.subplots(figsize=(10, 3))
x = np.array(habitaciones)

# Dibujamos el historiograma
ax.hist(x, linewidth=0.5, edgecolor="white")
plt.show()
print(habitaciones.max())

También se pueden mostrar gráficos con pandas. Son métodos más sencillos, pero dan menos juego
Además, parece que solo se pueden mostrar de uno en uno (y no en forma de cuadrícula como matplotlib)

In [ ]:
# --- Gráficos con pandas ---
# Gráfico 1
data["median_house_value"].plot.hist()
plt.figure()

# Gráfico 2
data.plot.scatter(x="median_income", y="median_house_value")
plt.figure()

# Gráfico 3
data.boxplot(column="population", grid=True)
plt.figure()

# Gráfico 4
data["ocean_proximity"].value_counts().plot.bar()
plt.figure() # El último se puede omitir con pandas

### Eliminamos outliers de todas las variables
Para cada variable, miramos cada uno de sus valores y eliminamos los que estén por muy alejados de la media (1.5 * RangoIntercuartil)

ERROR: Hay valores que, aunque están alejados de la mediana, no deberían considerarse *outliers*
Quizá hay que preprocesar algunas columnas de datos antes de aplicar el "algoritmo" de eliminación de outliers

In [ ]:
# --- Eliminación de outliers ---

separacion_maxima = 1.5
data_sin_outliers = data.copy()

# Solo nos interesan las variables numéricas, no categóricas
variables_numericas = data_sin_outliers.select_dtypes(include="number")
print(list(variables_numericas.columns))

for col in variables_numericas.columns:
    columnName = variables_numericas[col]
    # print("\n", col, "\n", column)

    # IQR o RI: Rango intercuartil
    Q1, Q3 = variables_numericas[col].quantile([0.25, 0.75])
    rango_intercuartil = Q3 - Q1
    print(col, Q1, Q3, rango_intercuartil)

    # Eliminamos los outliers por encima del valor especificado
    """ data_limpio = data[(data["columna"] >= Q1 - 1.5 * IQR) & (data["columna"] <= Q3 + 1.5 * IQR)] """
    # Para cambiar todas a la vez: data[~((data < (Q1 - 1.5 * IQR)) | (data > (Q3 + 1.5 * IQR))).any(axis=1)]
    data_sin_outliers = data_sin_outliers[
        (data_sin_outliers[col] >= Q1 - separacion_maxima * rango_intercuartil)
        & (data_sin_outliers[col] <= Q3 + separacion_maxima * rango_intercuartil)
    ]

In [ ]:
# Comprobamos si se han eliminado outliers de 'population' y 'median_house_value'

def printInfo(df, columnName):
    variable = df[columnName]
    print(f"{columnName.upper()}:")
    print(
        f"Mean: {variable.mean()}, Median: {variable.median()}, Min: {variable.min()}, Max: {variable.max()}"
    )
    print(f"Std: {variable.std()}")

printInfo(data, "population")
printInfo(data_sin_outliers, "population")
print()
printInfo(data, "median_house_value")
printInfo(data_sin_outliers, "median_house_value")

In [ ]:
# También podemos visualizar los datos
# plotGraphs(data)
# plotGraphs(data_sin_outliers)

In [ ]:
# Y como boxplots
plotBoxplots(data)
print("DATA SIN OUTLIERS")
plotBoxplots(data_sin_outliers)

In [ ]:
data_sin_outliers.describe()

# Notas aparte

In [ ]:
# NOTAS APARTE
# house = data.iloc[i, :] # Para mirar casa por casa